# Sequence, structure & data-visualisation utilities

Smaller standalone analyses: plotting IUPred2 intrinsic-disorder profiles along a sequence, computing per-amino-acid van der Waals volumes from Bondi radii, and formatting tabular NMR-derived measurements into annotated heatmaps.

**Contents**

1. [IUPred2 disorder profile](#sec1)
2. [Amino-acid van der Waals volumes](#sec2)
3. [NMR data heatmaps](#sec3)

> Notebooks were consolidated from separate working scripts; unused and broken scripts were removed. Each section keeps its own imports and reads independently. Cell outputs were cleared, and input data files are referenced by generic names (not included).


<a id="sec1"></a>
## 1. IUPred2 disorder profile

Plots an IUPred2 intrinsic-disorder prediction along a protein sequence, matched to a reference figure aspect ratio.

<sub>Source script: <code>iupred_disorder_profile.ipynb</code></sub>

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

ASPECT_RATIO = 437 / 137  # match reference figure

# load json
with open("IUPred_FUS_data.json") as f:
    data = json.load(f)

y = np.array(data["iupred2"])
x = np.arange(1, len(y) + 1)

# set figure with desired aspect ratio
height = 3
width = height * ASPECT_RATIO
plt.figure(figsize=(width, height))

# plot lines
plt.plot(x, y, color="red", lw=1, label="IUPred2 long disorder prediction")

# Add horizontal line at y=0.5
plt.axhline(y=0.5, color='gray', linestyle='--', lw=1, label='Disorder threshold ')

# axes styling
plt.xlim(1, len(y))
plt.ylim(0, 1)
plt.xlabel("Residue / Codon")
plt.ylabel("Score")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

ASPECT_RATIO = 437 / 137  # match reference figure

# load json
with open("IUPred_FMRP_data.json") as f:
    data = json.load(f)

y = np.array(data["iupred2"])
x = np.arange(1, len(y) + 1)

# set figure with desired aspect ratio
height = 3
width = height * ASPECT_RATIO
plt.figure(figsize=(width, height))

# plot lines
plt.plot(x, y, color="red", lw=1, label="IUPred2 long disorder prediction")

# Add horizontal line at y=0.5
plt.axhline(y=0.5, color='gray', linestyle='--', lw=1, label='Disorder threshold ')

# axes styling
plt.xlim(1, len(y))
plt.ylim(0, 1)
plt.xlabel("Residue / Codon")
plt.ylabel("Score")

plt.legend()
plt.tight_layout()
plt.show()

<a id="sec2"></a>
## 2. Amino-acid van der Waals volumes

Computes per-amino-acid van der Waals volumes from Bondi atomic radii and residue formulas.

<sub>Source script: <code>amino_acid_vdw_volumes.ipynb</code></sub>

In [ ]:
import pandas as pd
import math

# van der Waals radii (Å) from Bondi
r_vdw = {
    "H": 1.20,
    "C": 1.70,
    "N": 1.55,
    "O": 1.52,
    "S": 1.80,
    "P": 1.80
}

# Compute atomic volumes
v_vdw = {el: (4/3)*math.pi*(r**3) for el, r in r_vdw.items()}

# Molecular formulas for amino acids (neutral, uncharged side chains)
aa_formulas = {
    "Ala": {"C":3,"H":7,"N":1,"O":2},
    "Arg": {"C":6,"H":14,"N":4,"O":2},
    "Asn": {"C":4,"H":8,"N":2,"O":3},
    "Asp": {"C":4,"H":7,"N":1,"O":4},
    "Cys": {"C":3,"H":7,"N":1,"O":2,"S":1},
    "Gln": {"C":5,"H":10,"N":2,"O":3},
    "Glu": {"C":5,"H":9,"N":1,"O":4},
    "Gly": {"C":2,"H":5,"N":1,"O":2},
    "His": {"C":6,"H":9,"N":3,"O":2},
    "Ile": {"C":6,"H":13,"N":1,"O":2},
    "Leu": {"C":6,"H":13,"N":1,"O":2},
    "Lys": {"C":6,"H":14,"N":2,"O":2},
    "Met": {"C":5,"H":11,"N":1,"O":2,"S":1},
    "Phe": {"C":9,"H":11,"N":1,"O":2},
    "Pro": {"C":5,"H":9,"N":1,"O":2},
    "Ser": {"C":3,"H":7,"N":1,"O":3},
    "Thr": {"C":4,"H":9,"N":1,"O":3},
    "Trp": {"C":11,"H":12,"N":2,"O":2},
    "Tyr": {"C":9,"H":11,"N":1,"O":3},
    "Val": {"C":5,"H":11,"N":1,"O":2},
}

# Nucleotide formulas (RNA monophosphates, neutralized)
nt_formulas = {
    "AMP": {"C":10,"H":14,"N":5,"O":7,"P":1},
    "CMP": {"C":9,"H":14,"N":3,"O":8,"P":1},
    "GMP": {"C":10,"H":14,"N":5,"O":8,"P":1},
    "UMP": {"C":9,"H":13,"N":2,"O":9,"P":1},
}

def compute_volume(formula):
    return sum(v_vdw[el] * count for el, count in formula.items())

def sphere_radius_from_volume(V):
    return ((3 * V) / (4 * math.pi)) ** (1/3)

records = []
for name, formula in {**aa_formulas, **nt_formulas}.items():
    V = compute_volume(formula)
    Rh = sphere_radius_from_volume(V)
    records.append({"Molecule": name, "Volume_Å3": V, "R_h_Å": Rh})

df = pd.DataFrame(records)
print(df)


## Modify and Run PyMOL Script

### Subtask:
Provide the modified PyMOL script, and instruct the user on how to run it in the previously set up Conda environment to calculate and present the solvent-excluded surface volumes and equivalent hydrodynamic radii for amino acids and nucleotides.

#### Instructions
1. **Save the following modified Python script** (which uses `import pymol` instead of `from pymol2 import PyMOL` and adjusts the main execution block) into a file named `calculate_properties.py` on your local machine:

```python
import numpy as np
import pymol

AA_FRAGMENTS = [
    "ala","arg","asn","asp","cys","gln","glu","gly","his",
    "ile","leu","lys","met","phe","pro","ser","thr","trp","tyr","val"
]

# maps base → PyMOL fragment name
BASE_FRAG = {
    "A": "adenine",
    "C": "cytosine",
    "G": "guanine",
    "U": "uracil"
}

def build_nucleotide(cmd, base, name):
    """
    Build an RNA nucleotide monophosphate (e.g., AMP)
    using PyMOL's internal fragments.
    """
    base_frag = BASE_FRAG[base]

    # Load fragments
    cmd.fragment(base_frag, name=f"{name}_base")
    cmd.fragment("ribose", name=f"{name}_rib")
    cmd.fragment("phosphate", name=f"{name}_phos")

    # Fuse base ↔ ribose
    cmd.fuse(f"{name}_rib", f"{name}_base")
    cmd.delete(f"{name}_base")

    # Fuse ribose ↔ phosphate
    cmd.fuse(f"{name}_phos", f"{name}_rib")
    cmd.delete(f"{name}_rib")

    # Rename to final object
    cmd.set_name(f"{name}_phos", name)

def compute_ses_volume(cmd, obj, probe=1.4, quality=2):
    """
    Compute solvent-excluded (Connolly) volume using PyMOL's surface engine.
    """
    cmd.set("surface_quality", quality)
    cmd.set("solvent_radius", probe)
    cmd.show("surface", obj)
    vol_info = cmd.get_volume(obj)
    return vol_info["volume"]

def equivalent_hydrodynamic_radius(vol):
    return ( (3*vol) / (4*np.pi) ) ** (1/3)

# -------------------------------
# Main procedure
# -------------------------------

if __name__ == "__main__":
    # For a standard PyMOL Python installation (e.g., via conda-forge),
    # the 'cmd' object is directly accessible through 'pymol.cmd'.
    # We reinitialize the PyMOL session and then assign pymol.cmd to 'cmd'.
    pymol.cmd.reinitialize()
    cmd = pymol.cmd

    results = []

    # --- Amino acids ---
    for aa in AA_FRAGMENTS:
        cmd.fragment(aa, name=aa)
        V = compute_ses_volume(cmd, aa)
        Rh = equivalent_hydrodynamic_radius(V)
        results.append((aa.upper(), V, Rh))
        cmd.delete(aa)

    # --- Nucleotides: build AMP, CMP, GMP, UMP ---
    for base in ["A","C","G","U"]:
        name = base + "MP"
        build_nucleotide(cmd, base, name)
        V = compute_ses_volume(cmd, name)
        Rh = equivalent_hydrodynamic_radius(V)
        results.append((name, V, Rh))
        cmd.delete(name)

    # Print results
    print(f"{'Molecule':<8} {'Volume (Å^3)':>15} {'R_h (Å)':>10}")
    for mol, V, Rh in results:
        print(f"{mol:<8} {V:15.2f} {Rh:10.3f}")
```

2. **Open your terminal** where you previously set up the `pymol_env` Conda environment.
3. **Activate the Conda environment** by running `conda activate pymol_env`.
4. **Navigate to the directory** where you saved `calculate_properties.py`.
5. **Execute the script** using Python: `python calculate_properties.py`.
6. **Copy the entire output** (the printed table of Molecule, Volume, and R_h values) from your terminal and paste it back into this chat. This output will be used in the next step.


## Modify and Run PyMOL Script

### Subtask:
Provide the modified PyMOL script, and instruct the user on how to run it in the previously set up Conda environment to calculate and present the solvent-excluded surface volumes and equivalent hydrodynamic radii for amino acids and nucleotides.

#### Instructions
1. **Save the following modified Python script** (which uses `import pymol` instead of `from pymol2 import PyMOL` and adjusts the main execution block) into a file named `calculate_properties.py` on your local machine:

```python
import numpy as np
import pymol

AA_FRAGMENTS = [
    "ala","arg","asn","asp","cys","gln","glu","gly","his",
    "ile","leu","lys","met","phe","pro","ser","thr","trp","tyr","val"
]

# maps base → PyMOL fragment name
BASE_FRAG = {
    "A": "adenine",
    "C": "cytosine",
    "G": "guanine",
    "U": "uracil"
}

def build_nucleotide(cmd, base, name):
    """
    Build an RNA nucleotide monophosphate (e.g., AMP)
    using PyMOL's internal fragments.
    """
    base_frag = BASE_FRAG[base]

    # Load fragments
    cmd.fragment(base_frag, name=f"{name}_base")
    cmd.fragment("ribose", name=f"{name}_rib")
    cmd.fragment("phosphate", name=f"{name}_phos")

    # Fuse base ↔ ribose
    cmd.fuse(f"{name}_rib", f"{name}_base")
    cmd.delete(f"{name}_base")

    # Fuse ribose ↔ phosphate
    cmd.fuse(f"{name}_phos", f"{name}_rib")
    cmd.delete(f"{name}_rib")

    # Rename to final object
    cmd.set_name(f"{name}_phos", name)

def compute_ses_volume(cmd, obj, probe=1.4, quality=2):
    """
    Compute solvent-excluded (Connolly) volume using PyMOL's surface engine.
    """
    cmd.set("surface_quality", quality)
    cmd.set("solvent_radius", probe)
    cmd.show("surface", obj)
    vol_info = cmd.get_volume(obj)
    return vol_info["volume"]

def equivalent_hydrodynamic_radius(vol):
    return ( (3*vol) / (4*np.pi) ) ** (1/3)

# -------------------------------
# Main procedure
# -------------------------------

if __name__ == "__main__":
    # For a standard PyMOL Python installation (e.g., via conda-forge),
    # the 'cmd' object is directly accessible through 'pymol.cmd'.
    # We reinitialize the PyMOL session and then assign pymol.cmd to 'cmd'.
    pymol.cmd.reinitialize()
    cmd = pymol.cmd

    results = []

    # --- Amino acids ---
    for aa in AA_FRAGMENTS:
        cmd.fragment(aa, name=aa)
        V = compute_ses_volume(cmd, aa)
        Rh = equivalent_hydrodynamic_radius(V)
        results.append((aa.upper(), V, Rh))
        cmd.delete(aa)

    # --- Nucleotides: build AMP, CMP, GMP, UMP ---
    for base in ["A","C","G","U"]:
        name = base + "MP"
        build_nucleotide(cmd, base, name)
        V = compute_ses_volume(cmd, name)
        Rh = equivalent_hydrodynamic_radius(V)
        results.append((name, V, Rh))
        cmd.delete(name)

    # Print results
    print(f"{'Molecule':<8} {'Volume (Å^3)':>15} {'R_h (Å)':>10}")
    for mol, V, Rh in results:
        print(f"{mol:<8} {V:15.2f} {Rh:10.3f}")
```

2. **Open your terminal** where you previously set up the `pymol_env` Conda environment.
3. **Activate the Conda environment** by running `conda activate pymol_env`.
4. **Navigate to the directory** where you saved `calculate_properties.py`.
5. **Execute the script** using Python: `python calculate_properties.py`.
6. **Copy the entire output** (the printed table of Molecule, Volume, and R_h values) from your terminal and paste it back into this chat. This output will be used in the next step.


## Modify and Run PyMOL Script

### Subtask:
Provide the modified PyMOL script, and instruct the user on how to run it in the previously set up Conda environment to calculate and present the solvent-excluded surface volumes and equivalent hydrodynamic radii for amino acids and nucleotides.

#### Instructions
1. **Save the following modified Python script** (which uses `import pymol` instead of `from pymol2 import PyMOL` and adjusts the main execution block) into a file named `calculate_properties.py` on your local machine:

```python
import numpy as np
import pymol

AA_FRAGMENTS = [
    "ala","arg","asn","asp","cys","gln","glu","gly","his",
    "ile","leu","lys","met","phe","pro","ser","thr","trp","tyr","val"
]

# maps base → PyMOL fragment name
BASE_FRAG = {
    "A": "adenine",
    "C": "cytosine",
    "G": "guanine",
    "U": "uracil"
}

def build_nucleotide(cmd, base, name):
    """
    Build an RNA nucleotide monophosphate (e.g., AMP)
    using PyMOL's internal fragments.
    """
    base_frag = BASE_FRAG[base]

    # Load fragments
    cmd.fragment(base_frag, name=f"{name}_base")
    cmd.fragment("ribose", name=f"{name}_rib")
    cmd.fragment("phosphate", name=f"{name}_phos")

    # Fuse base ↔ ribose
    cmd.fuse(f"{name}_rib", f"{name}_base")
    cmd.delete(f"{name}_base")

    # Fuse ribose ↔ phosphate
    cmd.fuse(f"{name}_phos", f"{name}_rib")
    cmd.delete(f"{name}_rib")

    # Rename to final object
    cmd.set_name(f"{name}_phos", name)

def compute_ses_volume(cmd, obj, probe=1.4, quality=2):
    """
    Compute solvent-excluded (Connolly) volume using PyMOL's surface engine.
    """
    cmd.set("surface_quality", quality)
    cmd.set("solvent_radius", probe)
    cmd.show("surface", obj)
    vol_info = cmd.get_volume(obj)
    return vol_info["volume"]

def equivalent_hydrodynamic_radius(vol):
    return ( (3*vol) / (4*np.pi) ) ** (1/3)

# -------------------------------
# Main procedure
# -------------------------------

if __name__ == "__main__":
    # For a standard PyMOL Python installation (e.g., via conda-forge),
    # the 'cmd' object is directly accessible through 'pymol.cmd'.
    # We reinitialize the PyMOL session and then assign pymol.cmd to 'cmd'.
    pymol.cmd.reinitialize()
    cmd = pymol.cmd

    results = []

    # --- Amino acids ---
    for aa in AA_FRAGMENTS:
        cmd.fragment(aa, name=aa)
        V = compute_ses_volume(cmd, aa)
        Rh = equivalent_hydrodynamic_radius(V)
        results.append((aa.upper(), V, Rh))
        cmd.delete(aa)

    # --- Nucleotides: build AMP, CMP, GMP, UMP ---
    for base in ["A","C","G","U"]:
        name = base + "MP"
        build_nucleotide(cmd, base, name)
        V = compute_ses_volume(cmd, name)
        Rh = equivalent_hydrodynamic_radius(V)
        results.append((name, V, Rh))
        cmd.delete(name)

    # Print results
    print(f"{'Molecule':<8} {'Volume (Å^3)':>15} {'R_h (Å)':>10}")
    for mol, V, Rh in results:
        print(f"{mol:<8} {V:15.2f} {Rh:10.3f}")
```

2. **Open your terminal** where you previously set up the `pymol_env` Conda environment.
3. **Activate the Conda environment** by running `conda activate pymol_env`.
4. **Navigate to the directory** where you saved `calculate_properties.py`.
5. **Execute the script** using Python: `python calculate_properties.py`.
6. **Copy the entire output** (the printed table of Molecule, Volume, and R_h values) from your terminal and paste it back into this chat. This output will be used in the next step.

## Modify and Run PyMOL Script

### Subtask:
Provide the modified PyMOL script, and instruct the user on how to run it in the previously set up Conda environment to calculate and present the solvent-excluded surface volumes and equivalent hydrodynamic radii for amino acids and nucleotides.

#### Instructions
1. **Save the following modified Python script** (which uses `import pymol` instead of `from pymol2 import PyMOL` and adjusts the main execution block) into a file named `calculate_properties.py` on your local machine:

```python
import numpy as np
import pymol

AA_FRAGMENTS = [
    "ala","arg","asn","asp","cys","gln","glu","gly","his",
    "ile","leu","lys","met","phe","pro","ser","thr","trp","tyr","val"
]

# maps base → PyMOL fragment name
BASE_FRAG = {
    "A": "adenine",
    "C": "cytosine",
    "G": "guanine",
    "U": "uracil"
}

def build_nucleotide(cmd, base, name):
    """
    Build an RNA nucleotide monophosphate (e.g., AMP)
    using PyMOL's internal fragments.
    """
    base_frag = BASE_FRAG[base]

    # Load fragments
    cmd.fragment(base_frag, name=f"{name}_base")
    cmd.fragment("ribose", name=f"{name}_rib")
    cmd.fragment("phosphate", name=f"{name}_phos")

    # Fuse base ↔ ribose
    cmd.fuse(f"{name}_rib", f"{name}_base")
    cmd.delete(f"{name}_base")

    # Fuse ribose ↔ phosphate
    cmd.fuse(f"{name}_phos", f"{name}_rib")
    cmd.delete(f"{name}_rib")

    # Rename to final object
    cmd.set_name(f"{name}_phos", name)

def compute_ses_volume(cmd, obj, probe=1.4, quality=2):
    """
    Compute solvent-excluded (Connolly) volume using PyMOL's surface engine.
    """
    cmd.set("surface_quality", quality)
    cmd.set("solvent_radius", probe)
    cmd.show("surface", obj)
    vol_info = cmd.get_volume(obj)
    return vol_info["volume"]

def equivalent_hydrodynamic_radius(vol):
    return ( (3*vol) / (4*np.pi) ) ** (1/3)

# -------------------------------
# Main procedure
# -------------------------------

if __name__ == "__main__":
    # For a standard PyMOL Python installation (e.g., via conda-forge),
    # the 'cmd' object is directly accessible through 'pymol.cmd'.
    # We reinitialize the PyMOL session and then assign pymol.cmd to 'cmd'.
    pymol.cmd.reinitialize()
    cmd = pymol.cmd

    results = []

    # --- Amino acids ---
    for aa in AA_FRAGMENTS:
        cmd.fragment(aa, name=aa)
        V = compute_ses_volume(cmd, aa)
        Rh = equivalent_hydrodynamic_radius(V)
        results.append((aa.upper(), V, Rh))
        cmd.delete(aa)

    # --- Nucleotides: build AMP, CMP, GMP, UMP ---
    for base in ["A","C","G","U"]:
        name = base + "MP"
        build_nucleotide(cmd, base, name)
        V = compute_ses_volume(cmd, name)
        Rh = equivalent_hydrodynamic_radius(V)
        results.append((name, V, Rh))
        cmd.delete(name)

    # Print results
    print(f"{'Molecule':<8} {'Volume (Å^3)':>15} {'R_h (Å)':>10}")
    for mol, V, Rh in results:
        print(f"{mol:<8} {V:15.2f} {Rh:10.3f}")
```

2. **Open your terminal** where you previously set up the `pymol_env` Conda environment.
3. **Activate the Conda environment** by running `conda activate pymol_env`.
4. **Navigate to the directory** where you saved `calculate_properties.py`.
5. **Execute the script** using Python: `python calculate_properties.py`.
6. **Copy the entire output** (the printed table of Molecule, Volume, and R_h values) from your terminal and paste it back into this chat. This output will be used in the next step.


## Final Task

### Subtask:
Summarize the process, confirm PyMOL setup, and present the calculated volumes and hydrodynamic radii.


## Summary:

### Data Analysis Key Findings

*   **PyMOL Execution Constraint**: Direct execution of PyMOL within the Google Colab environment proved infeasible.
*   **Local Setup Requirement**: To proceed with the calculations, users were instructed to set up a dedicated `pymol_env` Conda environment locally on their machines, including PyMOL installation via `conda install -c conda-forge pymol`.
*   **Script Adaptation**: A Python script (`calculate_properties.py`) was provided, modified to be compatible with a standard `pymol` import (accessing commands via `pymol.cmd`) for execution in the locally established Conda environment.
*   **Calculated Properties**: The script is designed to compute solvent-excluded surface volumes (in Å$^3$) and equivalent hydrodynamic radii (in Å) for 20 standard amino acids and 4 common RNA nucleotides (AMP, CMP, GMP, UMP).

### Insights or Next Steps

*   The next crucial step is for the user to execute the provided script in their local PyMOL Conda environment and supply the generated output (table of volumes and radii) back to the agent for further analysis and presentation.
*   This approach highlights the necessity of bridging cloud-based development environments (like Colab) with specialized local software installations for tasks requiring specific computational tools.


<a id="sec3"></a>
## 3. NMR data heatmaps

Formats tabular NMR-derived values (pasted from a spreadsheet) into cleanly annotated heatmaps.

<sub>Source script: <code>nmr_data_heatmaps.ipynb</code></sub>

Paste your data below, replacing the placeholder data.

In [ ]:
import pandas as pd
import io

# Replace the placeholder data below with your data from Excel
excel_data = """
        A       C       G       U
A       3.76E-10        3.91E-10        3.72E-10        3.93E-10
C       3.62E-10        3.77E-10        3.58E-10        3.79E-10
D       3.52E-10        3.66E-10        3.48E-10        3.68E-10
E       3.44E-10        3.58E-10        3.41E-10        3.60E-10
F       3.33E-10        3.46E-10        3.30E-10        3.47E-10
G       4.03E-10        4.20E-10        3.99E-10        4.22E-10
H       3.25E-10        3.38E-10        3.22E-10        3.40E-10
I       3.39E-10        3.52E-10        3.36E-10        3.54E-10
K       3.15E-10        3.27E-10        3.13E-10        3.29E-10
L       3.31E-10        3.43E-10        3.28E-10        3.45E-10
M       3.48E-10        3.62E-10        3.45E-10        3.64E-10
N       3.54E-10        3.69E-10        3.51E-10        3.70E-10
P       3.74E-10        3.90E-10        3.71E-10        3.92E-10
Q       3.39E-10        3.53E-10        3.36E-10        3.55E-10
R       3.20E-10        3.32E-10        3.17E-10        3.34E-10
S       3.69E-10        3.84E-10        3.65E-10        3.86E-10
T       3.50E-10        3.65E-10        3.47E-10        3.66E-10
W       3.25E-10        3.37E-10        3.22E-10        3.39E-10
Y       3.21E-10        3.34E-10        3.18E-10        3.35E-10
V       3.34E-10        3.47E-10        3.31E-10        3.49E-10

"""

# Read the data into a pandas DataFrame, using the first row as header and first column as index
df = pd.read_csv(io.StringIO(excel_data), sep='\s+', header=0, index_col=0)


# Display the first few rows of the DataFrame to verify
display(df.head())

In [ ]:
import pandas as pd

# New data provided by the user
new_data = {'A': 8.749764166628866e-10, 'C': 8.13789254658489e-10, 'D': 7.70674592160026e-10, 'E': 7.412220599755665e-10, 'F': 6.947573935293368e-10, 'G': 1.0032057191048614e-10, 'H': 6.668874694336042e-10, 'I': 7.183448359022465e-10, 'K': 6.307418071336798e-10, 'L': 6.865596661720585e-10, 'M': 7.556614507543111e-10, 'N': 7.810192175581471e-10, 'P': 8.684467419116708e-10, 'Q': 7.205688137223773e-10, 'R': 6.465103523120218e-10, 'S': 8.432743725808979e-10, 'T': 7.656043645800258e-10, 'W': 6.649820766637938e-10, 'Y': 6.519432124154841e-10, 'V': 7.010353217841199e-10}

# Convert the dictionary to a pandas DataFrame and transpose it to a single column
df_new = pd.DataFrame.from_dict(new_data, orient='index')

# Display the new DataFrame
display(df_new)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# New data provided by the user
new_data = {'A': 8.749764166628866e-10, 'C': 8.13789254658489e-10, 'D': 7.70674592160026e-10, 'E': 7.412221e-10, 'F': 6.947574e-10, 'G': 1.003206e-9, 'H': 6.668875e-10, 'I': 7.183448e-10, 'K': 6.307418e-10, 'L': 6.865597e-10, 'M': 7.556615e-10, 'N': 7.810192e-10, 'P': 8.684467e-10, 'Q': 7.205688e-10, 'R': 6.465104e-10, 'S': 8.432744e-10, 'T': 7.656044e-10, 'W': 6.649821e-10, 'Y': 6.519432e-10, 'V': 7.010353e-10}

# Convert the dictionary to a pandas DataFrame and transpose it to a single column
df_new = pd.DataFrame.from_dict(new_data, orient='index')

# Map one-letter codes to three-letter codes
amino_acid_map = {
    'A': 'Ala', 'C': 'Cys', 'D': 'Asp', 'E': 'Glu', 'F': 'Phe', 'G': 'Gly',
    'H': 'His', 'I': 'Ile', 'K': 'Lys', 'L': 'Leu', 'M': 'Met', 'N': 'Asn',
    'P': 'Pro', 'Q': 'Gln', 'R': 'Arg', 'S': 'Ser', 'T': 'Thr', 'V': 'Val',
    'W': 'Trp', 'Y': 'Tyr'
}
# Replace the index with three-letter codes
df_new = df_new.rename(index=amino_acid_map)

# Generate the heatmap
plt.figure(figsize=(1.5, 8)) # Adjust figure size as needed
ax = sns.heatmap(df_new, annot=True, cmap="Reds", fmt=".2e", cbar=False) # Removed cbar_kws and set cbar=False

plt.title('Calculated diffusion coefficients')
plt.xlabel('Nucleotides')
plt.ylabel('Amino-acids')

plt.show()

# Display the new DataFrame (original, not scaled, as it might be used later)
display(df_new)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Generate the heatmap
plt.figure(figsize=(3, 8)) # Adjust figure size as needed
sns.heatmap(df_new, annot=True, cmap="Reds", fmt=".2e") # Add annotations and colormap, changed cmap to Reds

plt.title('New Data Heatmap') # Add a title
plt.xlabel('Nucleotides') # Add x-axis label
plt.ylabel('Amino-acids') # Add y-axis label

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Generate the heatmap using the divided DataFrame
plt.figure(figsize=(6, 8)) # Adjust figure size as needed
sns.heatmap(df_divided, annot=True, cmap="Reds", fmt=".2f", cbar=False) # Use the desired colormap and annotations, with cbar=False

plt.title('Maximal change in diffusion coefficients in percent') # Add a title
plt.xlabel('Nucleotides') # Add x-axis label
plt.ylabel('Amino-acids') # Add y-axis label

plt.show()

In [ ]:
import math

# Diffusion coefficients (m^2/s)
D_values = {
    'A': 8.749764166628866e-10, 'C': 8.13789254658489e-10, 'D': 7.70674592160026e-10,
    'E': 7.412221e-10, 'F': 6.947574e-10, 'G': 1.003206e-9, 'H': 6.668875e-10,
    'I': 7.183448e-10, 'K': 6.307418e-10, 'L': 6.865597e-10, 'M': 7.556615e-10,
    'N': 7.810192e-10, 'P': 8.684467e-10, 'Q': 7.205688e-10, 'R': 6.465104e-10,
    'S': 8.432744e-10, 'T': 7.656044e-10, 'W': 6.649821e-10, 'Y': 6.519432e-10,
    'V': 7.010353e-10
}

# Constants
kB = 1.380649e-23       # Boltzmann constant (J/K)
T = 298                 # Temperature (K)
eta = 9.32e-4           # Water viscosity (Pa·s)

def stokes_einstein_radius(D):
    """Return hydrodynamic radius (meters) from diffusion coefficient D."""
    return (kB * T) / (6 * math.pi * eta * D)

# Compute radii
radii = {aa: stokes_einstein_radius(D) for aa, D in D_values.items()}

# Print results
print("Hydrodynamic radii:")
for aa, r in radii.items():
    print(f"{aa}: {r:.3e} m ({r*1e9:.3f} nm)")
